# EDA de INEGI-GEO-001: Marco Geoestadístico 2025 de Sonora

Exploración de `data/raw/mg2025_26_sonora.zip`, el Marco Geoestadístico 2025 del INEGI para Sonora. Solo lectura y conteos: cada capa se lee directo del zip, sin extraerlo ni modificarlo, y sin limpieza de datos. Lo que salga de aquí sirve para el diccionario de datos y para elegir las capas del proyecto.

- Fuente: INEGI, Marco Geoestadístico 2025
- Descarga: https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551163061/26_sonora.zip
- Fecha de descarga: 13/10/2026

## 0. Preparación

Carga de librerías y de las rutas que ya define `health_data.pipeline`.

In [1]:
import hashlib
import re
import sys
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pyogrio
import shapely
from pyproj import CRS, Geod

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data.pipeline import RAW_DATA_DIR  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pio.renderers.default = "plotly_mimetype"  # guarda solo el JSON de cada figura, como en SALUD-001

Archivo crudo y su huella SHA-256, que debe empezar con `d7ba6400`.

In [2]:
ARCHIVO = RAW_DATA_DIR / "mg2025_26_sonora.zip"
sha256 = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
assert sha256.startswith("d7ba6400")

print(ARCHIVO.relative_to(ROOT).as_posix())
print(sha256)

data/raw/mg2025_26_sonora.zip
d7ba6400ba89a0925a38dd67a0d0228f5d99a47e135667c48105c4732bbf0188


Archivos del zip por carpeta.

In [3]:
zf = zipfile.ZipFile(ARCHIVO)
archivos = pd.Series([n for n in zf.namelist() if not n.endswith("/")])
archivos.str.split("/").str[0].value_counts()

conjunto_de_datos    80
catalogos            18
metadatos             3
Name: count, dtype: int64

Fecha de referencia del producto, tomada de `metadatos/mg_2025_26.txt` (viene en Latin-1).

In [4]:
metadato = zf.read("metadatos/mg_2025_26.txt").decode("latin-1")
re.search(r"Fecha de referencia[^:]*:\s*(\d{4}-\d{2}-\d{2})", metadato).group(1)

'2025-07-31'

Colores y funciones del notebook. Cada capa se lee directo del zip con `/vsizip/`; en los conteos, los vacíos aparecen como "(sin dato)".

In [5]:
AZUL, NARANJA, VERDE = "#2a78d6", "#eb6834", "#1baf7a"
CODIFICACION = "ISO-8859-1"  # la de los .dbf; se comprueba en la sección 4


def ruta(capa):
    return f"/vsizip/{ARCHIVO.as_posix()}/conjunto_de_datos/{capa}.shp"


def conteo(serie):
    return serie.fillna("(sin dato)").value_counts()

Hallazgo: la huella SHA-256 coincide y el zip trae 80 archivos de capas, 18 de catálogos y 3 de metadatos. La fecha de referencia es el 31/07/2025.

## 1. Inventario

Las 16 capas de `conjunto_de_datos/`, con la descripción de `catalogos/léeme.txt`. `pyogrio.read_info` lee solo los encabezados, sin cargar los registros.

In [6]:
DESCRIPCIONES = {
    "26ent": "Área geoestadística estatal",
    "26mun": "Áreas geoestadísticas municipales",
    "26ar": "AGEB rurales",
    "26a": "AGEB urbanas",
    "26l": "Localidades urbanas y rurales amanzanadas",
    "26lpr": "Localidades rurales puntuales",
    "26ti": "Territorio insular",
    "26pe": "Polígono externo de localidades rurales con caserío disperso",
    "26pem": "Polígono externo de manzana (caserío disperso urbano)",
    "26m": "Manzanas",
    "26fm": "Frentes de manzana",
    "26e": "Ejes de vialidad",
    "26cd": "Caserío disperso",
    "26sia": "Servicios de tipo área",
    "26sil": "Servicios de tipo línea",
    "26sip": "Servicios de tipo punto",
}
capas = sorted(Path(n).stem for n in zf.namelist() if n.lower().endswith(".shp"))
assert sorted(DESCRIPCIONES) == capas

filas = []
for capa in DESCRIPCIONES:
    info = pyogrio.read_info(ruta(capa), encoding=CODIFICACION)
    filas.append(
        {
            "capa": capa,
            "qué es": DESCRIPCIONES[capa],
            "geometría": info["geometry_type"],
            "registros": info["features"],
            "n_columnas": len(info["fields"]),
            "columnas": ", ".join(info["fields"]),
        }
    )
inventario = pd.DataFrame(filas).set_index("capa")
inventario

,qué es,geometría,registros,n_columnas,columnas
capa,,,,,
26ent,Área geoestadística estatal,Polygon,1,3,"CVEGEO, CVE_ENT, NOMGEO"
26mun,Áreas geoestadísticas municipales,Polygon,72,4,"CVEGEO, CVE_ENT, CVE_MUN, NOMGEO"
26ar,AGEB rurales,Polygon,1353,4,"CVEGEO, CVE_ENT, CVE_MUN, CVE_AGEB"
26a,AGEB urbanas,Polygon,3223,5,"CVE_ENT, CVE_MUN, CVE_LOC, CVE_AGEB, CVEGEO"
26l,Localidades urbanas y rurales amanzanadas,Polygon,955,6,"CVEGEO, CVE_ENT, CVE_MUN, CVE_LOC, NOMGEO, AMBITO"
26lpr,Localidades rurales puntuales,Point,16223,8,"CVEGEO, CVE_ENT, CVE_MUN, CVE_LOC, CVE_AGEB, CVE_MZA, NOMGEO, PLANO"
26ti,Territorio insular,Polygon,24,5,"CVEGEO, CVE_ENT, CVE_MUN, CVE_AGEB, NOMGEO"
26pe,Polígono externo de localidades rurales con caserío disperso,Polygon,836,4,"CVEGEO, CVE_ENT, CVE_MUN, CVE_LOC"
26pem,Polígono externo de manzana (caserío disperso urbano),Polygon,246,6,"CVEGEO, CVE_ENT, CVE_MUN, CVE_LOC, CVE_AGEB, CVE_MZA"


Capas por tipo de geometría.

In [7]:
inventario.groupby("geometría").agg(capas=("qué es", "size"), registros=("registros", "sum"))

,capas,registros
geometría,,
LineString,3,656885
MultiPoint,1,1043
Point,2,26340
Polygon,10,103444


Hallazgos:
- 16 capas: 10 de polígonos, 3 de líneas, 2 de puntos y 1 de multipuntos.
- Las más grandes son los frentes de manzana (423,805) y los ejes de vialidad (229,303). Las cinco capas principales son chicas: de 1 a 16,223 registros.
- En los .dbf las columnas AMBITO, CONDICION y GEOGRAFICO van sin acento.

## 2. Proyección

Nombre del sistema de coordenadas en el `.prj` de cada capa.

In [8]:
prj = pd.Series({capa: zf.read(f"conjunto_de_datos/{capa}.prj").decode("latin-1") for capa in capas})
nombre_prj = prj.str.extract(r'^PROJCS\["([^"]+)"', expand=False)
pd.DataFrame(
    {
        "capas": nombre_prj.groupby(nombre_prj).apply(lambda s: ", ".join(s.index)),
        "n": nombre_prj.value_counts(),
    }
)

,capas,n
MEXICO_ITRF_2008_LCC,"26a, 26cd, 26e, 26fm, 26l, 26m, 26pe, 26pem, 26sia, 26sil, 26sip",11
Mexico ITRF2008 / LCC,"26ar, 26ent, 26lpr, 26mun, 26ti",5


Parámetros de la proyección y del elipsoide de cada `.prj`, comparados con EPSG:6372 (Mexico ITRF2008 / LCC). Los `.prj` que traen `TOWGS84` se leen como un sistema con transformación, así que se toma su sistema base.

In [9]:
def parametros(crs):
    crs = crs.source_crs if crs.is_bound else crs
    valores = {p.name: p.value for p in crs.coordinate_operation.params}
    valores["Semieje mayor del elipsoide"] = crs.ellipsoid.semi_major_metre
    valores["Inverso del aplanamiento"] = crs.ellipsoid.inverse_flattening
    return valores


params = pd.DataFrame({capa: parametros(CRS.from_wkt(wkt)) for capa, wkt in prj.items()}).T.round(6)
epsg_6372 = pd.Series(parametros(CRS.from_epsg(6372))).round(6)

print("Juegos de parámetros distintos:", len(params.drop_duplicates()))
print("Capas con los mismos parámetros que EPSG:6372:", params.eq(epsg_6372).all(axis=1).sum())
pd.concat({"16 capas": params.drop_duplicates().iloc[0], "EPSG:6372": epsg_6372}, axis=1).map("{:,.12g}".format)

Juegos de parámetros distintos: 1
Capas con los mismos parámetros que EPSG:6372: 16


,16 capas,EPSG:6372
Latitude of false origin,12,12
Longitude of false origin,-102,-102
Latitude of 1st standard parallel,17.5,17.5
Latitude of 2nd standard parallel,29.5,29.5
Easting at false origin,"2,500,000","2,500,000"
Northing at false origin,0,0
Semieje mayor del elipsoide,"6,378,137","6,378,137"
Inverso del aplanamiento,298.257222,298.257222


Con los parámetros iguales, cada capa se lee con EPSG:6372 fijo y con la codificación de los `.dbf`.

In [10]:
def leer(capa):
    return gpd.read_file(ruta(capa), encoding=CODIFICACION).set_crs(6372, allow_override=True)

Hallazgo: 5 capas nombran la proyección "Mexico ITRF2008 / LCC" y 11 "MEXICO_ITRF_2008_LCC", pero las 16 tienen los mismos parámetros que EPSG:6372: paralelos 17.5 y 29.5, origen en 12 y −102, falso este 2,500,000 y el mismo elipsoide.

## 3. Capas principales

### 3.1 26ent: estado

Claves, nombre y validez de la geometría.

In [11]:
ent = leer("26ent")
ent.drop(columns="geometry").assign(valida=ent.is_valid)

,CVEGEO,CVE_ENT,NOMGEO,valida
0,26,26,Sonora,True


Área en km² con la proyección, que es cónica conforme y no conserva áreas, y área geodésica sobre el elipsoide GRS80, para ver la diferencia.

In [12]:
area_estado = ent.area.sum() / 1e6
area_geodesica = abs(Geod(ellps="GRS80").geometry_area_perimeter(ent.to_crs(4326).geometry.iloc[0])[0]) / 1e6

pd.Series(
    {
        "área en EPSG:6372 (km²)": area_estado,
        "área geodésica (km²)": area_geodesica,
        "diferencia (%)": (area_estado / area_geodesica - 1) * 100,
    }
).round(2)

área en EPSG:6372 (km²)    180633.84
área geodésica (km²)       180399.81
diferencia (%)                  0.13
dtype: float64

Límites del estado en grados (EPSG:4326).

In [13]:
limites = pd.Series(ent.to_crs(4326).total_bounds, index=["lon_min", "lat_min", "lon_max", "lat_max"])
limites.round(4)

lon_min   -115.0530
lat_min     26.2970
lon_max   -108.4243
lat_max     32.4939
dtype: float64

Hallazgos:
- Un solo polígono válido, con clave 26 y nombre Sonora.
- Área de 180,633.84 km² en EPSG:6372 y de 180,399.81 km² sobre el elipsoide: la proyección la agranda 0.13 %.
- Longitud de −115.0530 a −108.4243 y latitud de 26.2970 a 32.4939.

### 3.2 26mun: municipios

Número de municipios y claves contra la lista completa de 001 a 072. CVEGEO debe ser entidad + municipio.

In [14]:
mun = leer("26mun")
esperadas = {f"{i:03d}" for i in range(1, 73)}

pd.Series(
    {
        "municipios": len(mun),
        "claves faltantes": len(esperadas - set(mun["CVE_MUN"])),
        "claves sobrantes": len(set(mun["CVE_MUN"]) - esperadas),
        "claves repetidas": mun["CVE_MUN"].duplicated().sum(),
        "CVEGEO distinto de CVE_ENT + CVE_MUN": mun["CVEGEO"].ne(mun["CVE_ENT"] + mun["CVE_MUN"]).sum(),
    }
)

municipios                              72
claves faltantes                         0
claves sobrantes                         0
claves repetidas                         0
CVEGEO distinto de CVE_ENT + CVE_MUN     0
dtype: int64

Nombres con acento o ñ, para revisar que se leyeron bien.

In [15]:
con_acento = mun.loc[mun["NOMGEO"].str.contains(r"[^\x00-\x7f]"), ["CVE_MUN", "NOMGEO"]].set_index("CVE_MUN")
print("Nombres con acento o ñ:", len(con_acento))
con_acento

Nombres con acento o ñ: 22


,NOMGEO
CVE_MUN,
037,Mazatán
041,Nacozari de García
034,Huépac
014,Baviácora
016,Benjamín Hill
044,Ónavas
071,Benito Juárez
069,Yécora
055,San Luis Río Colorado


Validez de las geometrías.

In [16]:
mun.is_valid.value_counts()

True    72
Name: count, dtype: int64

Área de los municipios en km²: resumen, los 5 más grandes y los 5 más chicos.

In [17]:
mun["AREA_KM2"] = mun.area / 1e6
mun["AREA_KM2"].describe().round(1)

count       72.0
mean      2508.8
std       2801.6
min        151.6
25%        918.7
50%       1579.2
75%       2880.6
max      16908.0
Name: AREA_KM2, dtype: float64

In [18]:
orden = mun.sort_values("AREA_KM2", ascending=False)[["CVE_MUN", "NOMGEO", "AREA_KM2"]].round(1)
pd.concat(
    {"más grandes": orden.head(5).reset_index(drop=True), "más chicos": orden.tail(5).reset_index(drop=True)},
    axis=1,
)

más grandes                                 más chicos                       \
      CVE_MUN                 NOMGEO AREA_KM2    CVE_MUN               NOMGEO   
0         030             Hermosillo  16908.0        001              Aconchi   
1         017                Caborca  10732.8        071        Benito Juárez   
2         047              Pitiquito   9839.8        028             Granados   
3         055  San Luis Río Colorado   8986.1        007                 Atil   
4         029                Guaymas   7912.8        053  San Felipe de Jesús   

            
  AREA_KM2  
0    368.6  
1    366.9  
2    364.3  
3    302.1  
4    151.6

Suma de las áreas municipales contra el área del estado.

In [19]:
suma_municipios = mun["AREA_KM2"].sum()
pd.Series(
    {
        "suma de municipios (km²)": suma_municipios,
        "estado (km²)": area_estado,
        "diferencia (km²)": suma_municipios - area_estado,
        "diferencia (%)": (suma_municipios / area_estado - 1) * 100,
    }
).map("{:,.4f}".format)

suma de municipios (km²)    180,633.8375
estado (km²)                180,633.8375
diferencia (km²)                  0.0000
diferencia (%)                    0.0000
dtype: object

Hallazgos:
- 72 municipios con claves de 001 a 072, sin faltantes ni repetidas, y CVEGEO igual a entidad + municipio.
- 22 nombres con acento o ñ, todos bien leídos (Álamos, Ónavas, Puerto Peñasco).
- Las 72 geometrías son válidas. Hermosillo es el más grande (16,908.0 km²) y San Felipe de Jesús el más chico (151.6 km²).
- La suma de los municipios cuadra con el estado: la diferencia es 0.0000 km².

### 3.3 26l: localidades amanzanadas

Formato de CVEGEO (9 dígitos), repetidos y que sea entidad + municipio + localidad.

In [20]:
loc = leer("26l")
pd.Series(
    {
        "localidades": len(loc),
        "CVEGEO de 9 dígitos": loc["CVEGEO"].str.fullmatch(r"\d{9}").sum(),
        "CVEGEO repetidos": loc["CVEGEO"].duplicated().sum(),
        "CVEGEO distinto de la concatenación": loc["CVEGEO"]
        .ne(loc["CVE_ENT"] + loc["CVE_MUN"] + loc["CVE_LOC"])
        .sum(),
    }
)

localidades                            955
CVEGEO de 9 dígitos                    955
CVEGEO repetidos                         0
CVEGEO distinto de la concatenación      0
dtype: int64

Localidades por ámbito.

In [21]:
conteo(loc["AMBITO"])

AMBITO
Rural     851
Urbana    104
Name: count, dtype: int64

Validez de las geometrías.

In [22]:
loc.is_valid.value_counts()

True    955
Name: count, dtype: int64

Área por ámbito, en km².

In [23]:
loc.area.div(1e6).groupby(loc["AMBITO"]).describe().round(3)

,count,mean,std,min,25%,50%,75%,max
AMBITO,,,,,,,,
Rural,851.0,0.398,0.658,0.013,0.141,0.242,0.466,10.368
Urbana,104.0,8.170,23.442,0.181,0.865,1.788,4.109,204.718


Municipio donde cae un punto interior de cada localidad contra el municipio de su clave. La función sirve también para 26lpr.

In [24]:
MUNICIPIOS = mun[["CVE_MUN", "geometry"]].rename(columns={"CVE_MUN": "MUN_ESPACIAL"})


def ubicar(gdf):
    unido = gpd.sjoin(gdf[["CVE_MUN", "geometry"]], MUNICIPIOS, how="left", predicate="within")
    return pd.Series(
        np.select(
            [unido["MUN_ESPACIAL"].isna(), unido["MUN_ESPACIAL"].eq(unido["CVE_MUN"])],
            ["sin municipio", "en su municipio"],
            "en otro municipio",
        ),
        index=unido.index,
    )


ubicacion_l = ubicar(loc.assign(geometry=loc.representative_point()))
print("Filas después del cruce:", len(ubicacion_l))
ubicacion_l.value_counts()

Filas después del cruce: 955


en su municipio    955
Name: count, dtype: int64

Polígonos completos: cuántos salen de su municipio y por cuánta área. Se comparan con los catálogos del INEGI de localidades que rebasan su municipio y de localidades isla, filtrados a Sonora.

In [25]:
propio = gpd.GeoSeries(
    mun.set_index("CVE_MUN").geometry.reindex(loc["CVE_MUN"]).values, index=loc.index, crs=6372
)
fuera_m2 = loc.geometry.difference(propio).area

print("Polígonos que salen de su municipio:", (~loc.geometry.within(propio)).sum())
print("Con más de 1 m² fuera:", fuera_m2.gt(1).sum())


def catalogo(nombre):
    tabla = pd.read_csv(zf.open(f"catalogos/{nombre}.csv"), dtype=str)
    return tabla[tabla["CVE_ENT"] == "26"]


rebasan = pd.concat(
    [catalogo(f"localidades_{t}_que_rebasan_su_municipio") for t in ["urbanas", "rurales"]]
)
print("Localidades isla en Sonora:", len(catalogo("localidades_islas")))

loc.assign(
    m2_fuera=fuera_m2.round(1),
    pct_fuera=(fuera_m2 / loc.area * 100).round(3),
    en_catalogo=loc["CVEGEO"].isin(rebasan["CVE_ENT"] + rebasan["CVE_MUN"] + rebasan["CVE_LOC"]),
).loc[fuera_m2.gt(1), ["CVEGEO", "NOMGEO", "AMBITO", "m2_fuera", "pct_fuera", "en_catalogo"]]

Polígonos que salen de su municipio: 46
Con más de 1 m² fuera: 3
Localidades isla en Sonora: 0


,CVEGEO,NOMGEO,AMBITO,m2_fuera,pct_fuera,en_catalogo
166,260170001,Heroica Caborca,Urbana,56174.6,0.180,True
745,260430485,Campestre el Potrero,Rural,48213.2,16.669,True
853,260550001,San Luis Río Colorado,Urbana,1.6,0.000,False


Hallazgos:
- 955 localidades, 104 urbanas y 851 rurales, con CVEGEO de 9 dígitos, sin repetidos e igual a la concatenación de claves.
- Todas las geometrías son válidas. La mediana de área es 1.788 km² en las urbanas y 0.242 km² en las rurales.
- El punto interior de las 955 cae en el municipio de su clave.
- 46 polígonos salen de su municipio, pero solo 3 por más de 1 m². Heroica Caborca (56,174.6 m²) y Campestre el Potrero (48,213.2 m², el 16.669 % de su área) están en el catálogo del INEGI de localidades que rebasan su municipio; San Luis Río Colorado sale 1.6 m². En Sonora no hay localidades isla.

### 3.4 26lpr: localidades rurales puntuales

Estructura de CVEGEO: entidad (2) + municipio (3) + localidad (4) + AGEB (4, puede llevar letra) + manzana (3).

In [26]:
lpr = leer("26lpr")
concatenada = lpr["CVE_ENT"] + lpr["CVE_MUN"] + lpr["CVE_LOC"] + lpr["CVE_AGEB"] + lpr["CVE_MZA"]

pd.Series(
    {
        "localidades": len(lpr),
        "CVEGEO de 16 caracteres": lpr["CVEGEO"].str.len().eq(16).sum(),
        "con el patrón": lpr["CVEGEO"].str.fullmatch(r"\d{9}[0-9A-Z]{4}\d{3}").sum(),
        "igual a la concatenación": lpr["CVEGEO"].eq(concatenada).sum(),
        "AGEB con letra": lpr["CVE_AGEB"].str.contains("[A-Z]").sum(),
    }
)

localidades                 16223
CVEGEO de 16 caracteres     16223
con el patrón               16223
igual a la concatenación    16223
AGEB con letra               1161
dtype: int64

Valores de la clave de manzana.

In [27]:
conteo(lpr["CVE_MZA"])

CVE_MZA
800    16223
Name: count, dtype: int64

Localidades por PLANO.

In [28]:
conteo(lpr["PLANO"])

PLANO
No         15095
Si           851
Croquis      277
Name: count, dtype: int64

Claves repetidas: CVEGEO completo y clave de localidad de 9 dígitos.

In [29]:
lpr["CVELOC"] = lpr["CVEGEO"].str[:9]
pd.Series(
    {
        "CVEGEO repetidos": lpr["CVEGEO"].duplicated().sum(),
        "claves de 9 dígitos repetidas": lpr["CVELOC"].duplicated().sum(),
    }
)

CVEGEO repetidos                 0
claves de 9 dígitos repetidas    0
dtype: int64

Municipio donde cae cada punto contra el municipio de su clave.

In [30]:
ubicacion_lpr = ubicar(lpr)
print("Filas después del cruce:", len(ubicacion_lpr))
ubicacion_lpr.value_counts()

Filas después del cruce: 16223


en su municipio    16223
Name: count, dtype: int64

Localidades que están en 26l y en 26lpr, por clave de 9 dígitos.

In [31]:
en_l, en_lpr = set(loc["CVEGEO"]), set(lpr["CVELOC"])
pd.Series(
    {
        "en las dos capas": len(en_l & en_lpr),
        "solo en 26l": len(en_l - en_lpr),
        "solo en 26lpr": len(en_lpr - en_l),
        "localidades distintas": len(en_l | en_lpr),
    }
)

en las dos capas           851
solo en 26l                104
solo en 26lpr            15372
localidades distintas    16327
dtype: int64

Cruce de PLANO con la presencia en 26l, y de AMBITO de 26l con la presencia en 26lpr.

In [32]:
pd.crosstab(
    lpr["PLANO"],
    lpr["CVELOC"].isin(en_l).map({True: "en 26l", False: "no en 26l"}),
    margins=True,
    margins_name="Total",
)

CVELOC,en 26l,no en 26l,Total
PLANO,,,
Croquis,0,277,277
No,0,15095,15095
Si,851,0,851
Total,851,15372,16223


In [33]:
pd.crosstab(
    loc["AMBITO"],
    loc["CVEGEO"].isin(en_lpr).map({True: "en 26lpr", False: "no en 26lpr"}),
    margins=True,
    margins_name="Total",
)

CVEGEO,en 26lpr,no en 26lpr,Total
AMBITO,,,
Rural,851,0,851
Urbana,0,104,104
Total,851,104,955


Mapa de las localidades distintas: las de 26l con un punto interior de su polígono y las de 26lpr que no están en 26l. Los límites municipales están simplificados a 1 km, solo para dibujarlos.

In [34]:
CENTRO = {"lat": limites[["lat_min", "lat_max"]].mean(), "lon": limites[["lon_min", "lon_max"]].mean()}
fronteras = gpd.GeoSeries([shapely.line_merge(mun.boundary.union_all())], crs=6372)  # cada frontera, una vez
CAPA_LIMITES = {
    "source": fronteras.simplify(1000).to_crs(4326).set_precision(1e-4).__geo_interface__,
    "type": "line",
    "color": "#5f5e5a",
    "line": {"width": 0.7},
}
COLORES_TIPO = {
    "Rural puntual (26lpr)": VERDE,
    "Rural amanzanada (26l)": NARANJA,
    "Urbana (26l)": AZUL,
}

puntos = pd.concat(
    [
        loc.assign(
            geometry=loc.representative_point(),
            CLAVE=loc["CVEGEO"],
            TIPO=loc["AMBITO"].map({"Urbana": "Urbana (26l)", "Rural": "Rural amanzanada (26l)"}),
        ),
        lpr[~lpr["CVELOC"].isin(en_l)].assign(CLAVE=lambda d: d["CVELOC"], TIPO="Rural puntual (26lpr)"),
    ]
)[["CLAVE", "NOMGEO", "TIPO", "geometry"]].to_crs(4326)
puntos = puntos.assign(
    lat=puntos.geometry.y.round(5),
    lon=puntos.geometry.x.round(5),
    ETIQUETA=puntos["NOMGEO"] + " (" + puntos["CLAVE"] + ")",
).drop(columns="geometry")
print("Puntos en el mapa:", len(puntos))

fig = px.scatter_map(
    puntos,
    lat="lat",
    lon="lon",
    color="TIPO",
    color_discrete_map=COLORES_TIPO,
    category_orders={"TIPO": list(COLORES_TIPO)},
    hover_name="ETIQUETA",
    center=CENTRO,
    zoom=5.2,
    map_style="carto-positron",
    height=650,
    title="Localidades de Sonora por tipo, con límites municipales",
)
fig.update_traces(hovertemplate="%{hovertext}")  # nombre y clave; el recuadro lateral muestra el tipo
fig.for_each_trace(lambda t: t.update(marker_size=4 if t.name.startswith("Rural puntual") else 7))
fig.update_layout(map_layers=[CAPA_LIMITES], legend_title_text=None)
fig.show()

Puntos en el mapa: 16327


Hallazgos:
- 16,223 puntos con CVEGEO de 16 caracteres igual a la concatenación de claves. 1,161 claves de AGEB llevan letra y la clave de manzana siempre es 800.
- PLANO: 15,095 No, 851 Si y 277 Croquis. No hay CVEGEO ni claves de localidad repetidas.
- Los 16,223 puntos caen en el municipio de su clave.
- Las 851 localidades con PLANO = Si son las 851 rurales de 26l, y las 104 urbanas solo están en 26l. En total hay 16,327 localidades distintas: 104 solo en 26l, 851 en las dos capas y 15,372 solo en 26lpr.

### 3.5 26sip: servicios puntuales

Servicios por tipo geográfico (GEOGRAFICO).

In [35]:
sip = leer("26sip")
geograficos = conteo(sip["GEOGRAFICO"])
print("Servicios:", len(sip), "| tipos geográficos:", len(geograficos))
geograficos

Servicios: 10117 | tipos geográficos: 17


GEOGRAFICO
Escuela                               3450
Templo                                2073
Instalación Deportiva o Recreativa    1628
Tanque de Agua                         793
Centro de Asistencia Médica            681
Plaza                                  577
Pozo                                   244
Cementerio                             205
Instalación Gubernamental              160
Instalación de Comunicación             79
Instalación Diversa                     58
Estación de Transporte Terrestre        49
Instalación de Servicios                40
Mercado                                 38
Centro Comercial                        35
Aeródromo Civil                          4
Subestación Eléctrica                    3
Name: count, dtype: int64

Las 15 clasificaciones (TIPO) más comunes.

In [36]:
tipos = conteo(sip["TIPO"])
print("Valores distintos de TIPO:", len(tipos))
tipos.head(15)

Valores distintos de TIPO: 51


TIPO
No Aplica              2931
Primaria               1440
Preescolar             1246
Parque                  853
Tanque Elevado          656
Centro de Salud         583
Secundaria              510
Cancha                  268
Estadio                 220
Agua                    216
Jardín                  204
Palacio de Gobierno     152
Caja de Agua            137
Media Superior          119
Superior                 95
Name: count, dtype: int64

Cruce de condición y ámbito.

In [37]:
pd.crosstab(sip["CONDICION"], sip["AMBITO"], margins=True, margins_name="Total")

AMBITO,Rural,Urbana,Total
CONDICION,,,
En Construcción,6,11,17
En Operación,3571,5524,9095
Fuera de Uso,24,1,25
No Aplica,364,616,980
Total,3965,6152,10117


Nombres vacíos en NOMSERV: nulos, en blanco o con el valor "Ninguno", por tipo geográfico.

In [38]:
nombre = sip["NOMSERV"]
sin_nombre = nombre.isna() | nombre.str.strip().eq("") | nombre.eq("Ninguno")
print("Nulos:", nombre.isna().sum(), "| en blanco:", nombre.str.strip().eq("").sum(), "| Ninguno:", nombre.eq("Ninguno").sum())

nombres = pd.crosstab(sip["GEOGRAFICO"], sin_nombre.map({True: "sin nombre", False: "con nombre"}))
nombres = nombres.assign(Total=nombres.sum(axis=1)).sort_values("Total", ascending=False)
nombres.loc["Total"] = nombres.sum()
nombres

Nulos: 0 | en blanco: 0 | Ninguno: 6467


NOMSERV,con nombre,sin nombre,Total
GEOGRAFICO,,,
Escuela,2322,1128,3450
Templo,719,1354,2073
Instalación Deportiva o Recreativa,178,1450,1628
Tanque de Agua,24,769,793
Centro de Asistencia Médica,202,479,681
Plaza,57,520,577
Pozo,13,231,244
Cementerio,20,185,205
Instalación Gubernamental,8,152,160


Claves de los servicios. CVEGEO llega hasta la manzana, así que varios servicios pueden compartirlo; CVE_SERV los distingue dentro de la misma clave.

In [39]:
print("CVEGEO repetidos:", sip["CVEGEO"].duplicated().sum())
print("CVEGEO + CVE_SERV repetidos:", sip.duplicated(["CVEGEO", "CVE_SERV"]).sum())
pd.crosstab(sip["CVEGEO"].str.len().rename("largo de CVEGEO"), sip["AMBITO"], margins=True, margins_name="Total")

CVEGEO repetidos: 2013
CVEGEO + CVE_SERV repetidos: 0


AMBITO,Rural,Urbana,Total
largo de CVEGEO,,,
9,222,43,265
16,3743,6109,9852
Total,3965,6152,10117


Centros de Asistencia Médica por tipo y ámbito.

In [40]:
cam = sip[sip["GEOGRAFICO"] == "Centro de Asistencia Médica"]
pd.crosstab(cam["TIPO"], cam["AMBITO"], margins=True, margins_name="Total")

AMBITO,Rural,Urbana,Total
TIPO,,,
Centro de Rehabilitación,1,10,11
Centro de Salud,318,265,583
Hospital,0,87,87
Total,319,362,681


Centros de Asistencia Médica por tipo y condición.

In [41]:
pd.crosstab(cam["TIPO"], cam["CONDICION"], margins=True, margins_name="Total")

CONDICION,En Operación,Fuera de Uso,Total
TIPO,,,
Centro de Rehabilitación,11,0,11
Centro de Salud,580,3,583
Hospital,87,0,87
Total,678,3,681


Centros de Asistencia Médica por municipio y ámbito, con los 72 municipios.

In [42]:
cam_mun = (
    pd.crosstab(cam["CVE_MUN"], cam["AMBITO"])
    .reindex(mun["CVE_MUN"], fill_value=0)[["Urbana", "Rural"]]
    .assign(Total=lambda t: t.sum(axis=1))
    .join(mun.set_index("CVE_MUN")["NOMGEO"])
    .set_index("NOMGEO", append=True)
    .sort_values("Total", ascending=False)
)
print("Municipios sin centros:", cam_mun["Total"].eq(0).sum())
print("Municipios con un solo centro:", cam_mun["Total"].eq(1).sum())
cam_mun

Municipios sin centros: 0
Municipios con un solo centro: 14


,,Urbana,Rural,Total
CVE_MUN,NOMGEO,,,
030,Hermosillo,48,19,67
018,Cajeme,36,20,56
042,Navojoa,16,35,51
029,Guaymas,24,17,41
033,Huatabampo,7,33,40
003,Álamos,4,30,34
026,Etchojoa,11,21,32
055,San Luis Río Colorado,24,2,26
017,Caborca,12,11,23


Mapa de los Centros de Asistencia Médica por ámbito.

In [43]:
cam_mapa = cam.to_crs(4326).merge(mun[["CVE_MUN", "NOMGEO"]].rename(columns={"NOMGEO": "MUNICIPIO"}), on="CVE_MUN")
cam_mapa = cam_mapa.assign(lat=cam_mapa.geometry.y.round(5), lon=cam_mapa.geometry.x.round(5))

fig = px.scatter_map(
    cam_mapa.drop(columns="geometry"),
    lat="lat",
    lon="lon",
    color="AMBITO",
    color_discrete_map={"Urbana": AZUL, "Rural": NARANJA},
    category_orders={"AMBITO": ["Urbana", "Rural"]},
    hover_name="TIPO",
    hover_data={"NOMSERV": True, "MUNICIPIO": True, "AMBITO": False, "lat": False, "lon": False},
    center=CENTRO,
    zoom=5.2,
    map_style="carto-positron",
    height=650,
    title="Centros de Asistencia Médica de 26sip por ámbito",
)
fig.update_traces(marker_size=8)
fig.update_layout(map_layers=[CAPA_LIMITES], legend_title_text=None)
fig.show()

Hallazgos:
- 10,117 servicios de 17 tipos geográficos; los más comunes son escuelas (3,450) y templos (2,073). TIPO tiene 51 valores y el más común es "No Aplica" (2,931).
- 9,095 en operación, 980 con condición "No Aplica", 25 fuera de uso y 17 en construcción; 6,152 urbanos y 3,965 rurales.
- NOMSERV no tiene nulos ni blancos, pero 6,467 dicen "Ninguno".
- CVEGEO se repite 2,013 veces porque llega hasta la manzana; CVEGEO + CVE_SERV no se repite. 265 servicios traen solo la clave de localidad (9 caracteres).
- 681 Centros de Asistencia Médica: 362 urbanos y 319 rurales. Son 583 centros de salud, 87 hospitales (todos urbanos) y 11 centros de rehabilitación; 678 en operación y 3 centros de salud fuera de uso. 479 no tienen nombre.
- Los 72 municipios tienen al menos un centro. Hermosillo tiene 67 y 14 municipios tienen uno solo.

## 4. Calidad

Resumen de las cinco capas: geometrías vacías o nulas, inválidas, claves repetidas y nombres mal codificados (con Ã, Â o el carácter de reemplazo �).

In [44]:
MAL_CODIFICADO = r"Ã|Â|\ufffd"
PRINCIPALES = {
    "26ent": (ent, "NOMGEO"),
    "26mun": (mun, "NOMGEO"),
    "26l": (loc, "NOMGEO"),
    "26lpr": (lpr, "NOMGEO"),
    "26sip": (sip, "NOMSERV"),
}

pd.DataFrame(
    {
        capa: {
            "registros": len(gdf),
            "geometrías vacías o nulas": (gdf.geometry.isna() | gdf.geometry.is_empty).sum(),
            "geometrías inválidas": (~gdf.is_valid).sum(),
            "CVEGEO repetidos": gdf["CVEGEO"].duplicated().sum(),
            "nombres con acento o ñ": gdf[col].str.contains(r"[^\x00-\x7f]").sum(),
            "nombres mal codificados": gdf[col].str.contains(MAL_CODIFICADO).sum(),
        }
        for capa, (gdf, col) in PRINCIPALES.items()
    }
).T

,registros,geometrías vacías o nulas,geometrías inválidas,CVEGEO repetidos,nombres con acento o ñ,nombres mal codificados
26ent,1,0,0,0,0,0
26mun,72,0,0,0,22,0
26l,955,0,0,0,265,0
26lpr,16223,0,0,0,4103,0
26sip,10117,0,0,2013,1192,0


Codificación por capa: texto del `.cpg`, codificación que detecta GDAL y nombres mal codificados al leer solo la columna de nombres, con la codificación por omisión y con ISO-8859-1.

In [45]:
filas = []
for capa, (_, col) in PRINCIPALES.items():
    cpg = next(n for n in zf.namelist() if n.lower() == f"conjunto_de_datos/{capa}.cpg")
    fila = {"cpg": zf.read(cpg).decode("latin-1").strip(), "detectada": pyogrio.read_info(ruta(capa))["encoding"]}
    for etiqueta, codificacion in [("por omisión", None), (CODIFICACION, CODIFICACION)]:
        nombres = pyogrio.read_dataframe(ruta(capa), columns=[col], read_geometry=False, encoding=codificacion)[col]
        fila[f"mal codificados ({etiqueta})"] = nombres.str.contains(MAL_CODIFICADO).sum()
    filas.append(pd.Series(fila, name=capa))
pd.DataFrame(filas)

,cpg,detectada,mal codificados (por omisión),mal codificados (ISO-8859-1)
26ent,ISO 88591,ISO-8859-1,0,0
26mun,ISO 88591,ISO-8859-1,0,0
26l,ISO 88591,ISO-8859-1,0,0
26lpr,ISO 88591,ISO-8859-1,0,0
26sip,ISO 88591,ISO-8859-1,0,0


Texto del `.cpg` en las 16 capas.

In [46]:
cpg = pd.Series(
    {Path(n).stem: zf.read(n).decode("latin-1").strip() for n in zf.namelist() if n.lower().endswith(".cpg")}
)
cpg.groupby(cpg).apply(lambda s: ", ".join(s.index)).to_frame("capas").assign(n=cpg.value_counts())

,capas,n
88591,"26cd, 26e, 26fm, 26m, 26pe, 26pem",6
ISO 88591,"26ar, 26ent, 26l, 26lpr, 26mun, 26sia, 26sil, 26sip, 26ti",9
UTF-8,26a,1


Hallazgos:
- Ninguna de las cinco capas tiene geometrías vacías, nulas o inválidas.
- No hay CVEGEO repetidos, salvo en 26sip, donde se espera (2,013, ver 3.5).
- No hay nombres mal codificados. GDAL detecta ISO-8859-1 a partir del .cpg, y leer con ISO-8859-1 explícito da lo mismo. Hay 22 nombres con acento en 26mun, 265 en 26l, 4,103 en 26lpr y 1,192 en 26sip.
- Los .cpg no son uniformes: 9 capas dicen "ISO 88591", 6 "88591" y 1 "UTF-8". Por eso conviene fijar la codificación al leer.

## 5. Cierre

Comparación con las cifras de una revisión previa.

In [47]:
comparacion = [
    ("municipios", len(mun), 72),
    ("localidades amanzanadas (26l)", len(loc), 955),
    ("26l urbanas", loc["AMBITO"].eq("Urbana").sum(), 104),
    ("26l rurales", loc["AMBITO"].eq("Rural").sum(), 851),
    ("localidades puntuales (26lpr)", len(lpr), 16223),
    ("PLANO = No", lpr["PLANO"].eq("No").sum(), 15095),
    ("PLANO = Si", lpr["PLANO"].eq("Si").sum(), 851),
    ("PLANO = Croquis", lpr["PLANO"].eq("Croquis").sum(), 277),
    ("en 26l y en 26lpr", len(en_l & en_lpr), 851),
    ("localidades distintas", len(en_l | en_lpr), 16327),
    ("servicios (26sip)", len(sip), 10117),
    ("tipos de servicio", sip["GEOGRAFICO"].nunique(), 17),
    ("Centros de Asistencia Médica", len(cam), 681),
    ("centros urbanos", cam["AMBITO"].eq("Urbana").sum(), 362),
    ("centros rurales", cam["AMBITO"].eq("Rural").sum(), 319),
]
pd.DataFrame(comparacion, columns=["cifra", "archivo", "referencia"]).set_index("cifra").assign(
    diferencia=lambda t: t["archivo"] - t["referencia"]
)

,archivo,referencia,diferencia
cifra,,,
municipios,72,72,0
localidades amanzanadas (26l),955,955,0
26l urbanas,104,104,0
26l rurales,851,851,0
localidades puntuales (26lpr),16223,16223,0
PLANO = No,15095,15095,0
PLANO = Si,851,851,0
PLANO = Croquis,277,277,0
en 26l y en 26lpr,851,851,0


Hallazgos:
- Todas las cifras de la revisión previa coinciden con el archivo.
- Las cinco capas están limpias: geometrías válidas, claves completas y sin repetir (en 26sip, con CVEGEO + CVE_SERV) y nombres bien leídos.
- Cada localidad cae en el municipio de su clave. Solo 2 polígonos rebasan su municipio de forma visible, y el INEGI ya los reporta.
- Las 16 capas usan la proyección de EPSG:6372, aunque sus .prj la nombran distinto.

Capas que se usarán en el proyecto:
- 26mun: límites y mapas por municipio.
- 26l y 26lpr: ubicación de las 16,327 localidades, unidas por la clave de 9 dígitos (punto interior del polígono en 26l y punto en 26lpr).
- 26ent: contorno del estado.

26sip y las demás capas no se usan por ahora.